## Computational Physics -- Problem Set 8
##### _Theodore Landa_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

This was pretty fun, but also took a while. I reworked my leapfrog code from last time to make it more customizable and general. The plotting was also a bit tricky. The first plot is the one that was required, the second plot has a few more orbits on it to fill out the top spaces (for fun? Not sure but I wanted to see), and the third plot is some orbits very close to the separatrix (also just to see).

In [ ]:
def leapfrog_start(f, t0, xs, step):
    return xs + (step / 2) * f(xs, t0)

def leapfrog_step(f, xs1, xs2, t, step):
    xs = xs1 + step * f(xs2, t + step / 2)
    xs_next = xs2 + step * f(xs, t + step)
    return xs, xs_next

def default_cond(orbit, ts, xs):
    return False

def leapfrog(f, rng, xs_i, step, cond=default_cond): # This is a normal leapfrog with a conditional to halt integration
    ts = np.arange(rng[0], rng[1], step)
    orbit = np.zeros((ts.shape[0], *xs_i.shape))
    xs = xs_i.copy()
    xs_next = leapfrog_start(f, rng[0], xs, step)
    n = ts.shape[0]
    for i in range(ts.shape[0]):
        orbit[i] = xs
        xs, xs_next = leapfrog_step(f, xs, xs_next, ts[i], step)
        if cond(orbit, ts, xs):
            orbit[i+1] = xs
            n = i+2
            break
    return orbit[:n], ts[:n]

In [ ]:
g = 9.81
l = 9.81 # haha

def pend_orbit(orbit, ts, xs):
    return (xs[0] > np.pi or xs[1] < 0)

def pend(xs, t):
    return np.array([xs[1], 0-(g/l)*np.sin(xs[0])])

In [ ]:
for i in range(13):
    xs_i = np.array([0, np.sqrt(g/l)*0.2*i])
    orbit, _ = leapfrog(pend, [0, 400], xs_i, 0.001*np.sqrt(g/l), pend_orbit)
    for j in range(3):
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=1)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=1)
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=1)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=1)

plt.xlabel(r"angle $\theta$ [rad]")
plt.ylabel(r"angular velocity $\omega$ [rad s$^{-1}$]")
plt.title("Pendulum phase space orbits!")
plt.xlim(0-2*np.pi, 2*np.pi)
plt.ylim(0-2.5*np.sqrt(g/l), 2.5*np.sqrt(g/l))
plt.show()

In [ ]:
for i in range(16):
    xs_i = np.array([0, np.sqrt(g/l)*0.2*i])
    orbit, _ = leapfrog(pend, [0, 400], xs_i, 0.001*np.sqrt(g/l), pend_orbit)
    for j in range(3):
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=1)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=1)
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=1)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=1)

plt.xlabel(r"angle $\theta$ [rad]")
plt.ylabel(r"angular velocity $\omega$ [rad s$^{-1}$]")
plt.title("Pendulum phase space orbits!")
plt.xlim(0-2*np.pi, 2*np.pi)
plt.ylim(0-2.5*np.sqrt(g/l), 2.5*np.sqrt(g/l))
plt.savefig("pend_phase_space.svg", format="svg")
plt.savefig("pend_phase_space.png", format="png")
plt.show()

In [ ]:
separatrix_close = 0.002
num_close_orbs = 3
for i in range(num_close_orbs*2+1):
    xs_i = np.array([0, np.sqrt(g/l)*(2 + separatrix_close*(i-num_close_orbs))])
    orbit, _ = leapfrog(pend, [0, 400], xs_i, 0.001*np.sqrt(g/l), pend_orbit)
    for j in range(3):
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=0.3)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, orbit[:,1], 'k', linewidth=0.3)
        plt.plot(orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=0.3)
        plt.plot(0-orbit[:,0] + (j-1)*2*np.pi, 0-orbit[:,1], 'k', linewidth=0.3)
plt.xlabel(r"angle $\theta$ [rad]")
plt.ylabel(r"angular velocity $\omega$ [rad s$^{-1}$]")
plt.title("Pendulum phase space orbits close to separatrix!")
plt.savefig("pend_phase_space_separatrix.svg", format="svg")
plt.savefig("pend_phase_space_separatrix.png", format="png")
plt.xlim(0-2*np.pi, 2*np.pi)
plt.ylim(0-2.5*np.sqrt(g/l), 2.5*np.sqrt(g/l))
plt.show()

### Problem 2.

_(a)_ We begin with $\mathbf{F}=\mathbf{m\,a}$ where $\mathbf{F}$ is the sum of the forces on the golf ball; the relevant forces here are drag and gravity. The force of drag $\mathbf{F_{drag}}=-\frac{1}{2}\,\pi\,R^2\,\rho\,C\,v^2\,\mathbf{\hat{v}}$ is in the direction $-\mathbf{\hat{v}}=-(\dot{x},\dot{y})/\sqrt{\dot{x}^2+\dot{y}^2}$, and the force of gravity is just $\mathbf{F_{G}}=-m\,g\,\mathbf{\hat{y}}$. Therefore, componentwise, we have $$F_x=-\frac{1}{2}\,\pi\,R^2\,\rho\,C\,(\dot{x}^2+\dot{y}^2)\,\dot{x}/\sqrt{\dot{x}^2+\dot{y}^2}$$ and $$F_y=-m\,g-\frac{1}{2}\,\pi\,R^2\,\rho\,C\,(\dot{x}^2+\dot{y}^2)\,\dot{y}/\sqrt{\dot{x}^2+\dot{y}^2}$$ which simplify to $$\ddot{x}=-\frac{\pi\,R^2\,\rho\,C}{2\,m}\,\dot{x}\,\sqrt{\dot{x}^2+\dot{y}^2}$$ and $$\ddot{y}=-g-\frac{\pi\,R^2\,\rho\,C}{2\,m}\,\dot{y}\,\sqrt{\dot{x}^2+\dot{y}^2}$$

_(b)_ Plot!

_(c)_ In the trajectory from _(b)_, we got about `272.54`m traveled over horizontal ground. Keeping the same launch angle, velocity, and drag, but varying the mass, we can see that increasing the mass increases the distance traveled, with diminishing returns (big surprise). I plotted a bunch of trajectories with linearly increasing masses, and also the distance traveled as a function of mass (which is a way more informative plot!). 

_(d)_ I used my bisection algorithm from problem set 2 to compute the optimal launch angle for distance traveled (when launching at mach 0.5); the distance traveled at that optimal launch angle is `393.84`m. Apparently, the longest drive ever hit is `471`m; I'm pretty sure that includes the ball bouncing and rolling (we don't have that). Also, the drag coefficient apparently varies with the velocity, and there's also possibly a high velocity drag regime with a turbulent wake that lets the ball move faster; we haven't accounted for anything like that. People do be hitting golf balls very very far!!! Close to the physical limit, for sure.

In [ ]:
m_0 = 0.04593
m = m_0
R = 0.02134
rho = 1.22
C = 0.19
v_0 = 76  # this number from Claude for PGA tour. It SHOULD go 282yds = 257m
# ISSUES: dynamical reynolds number AND torque/spin/magnus effect.
# v_0 = 94
theta_0 = np.pi/6
xs_i_golf = np.array([0, 0, v_0*np.cos(theta_0), v_0*np.sin(theta_0)])

def golf(xs, t):
    const = (np.pi*R**2*rho*C/(2*m))*np.sqrt(xs[2]**2+xs[3]**2)
    return np.array([xs[2], xs[3], 0-const*xs[2], 0-g-const*xs[3]])

def golf_cond(orbit, ts, xs):
    return (xs[1] < 0)

In [ ]:
golf_orbit, golf_ts = leapfrog(golf, [0, 200], xs_i_golf, 0.001, golf_cond)
print(golf_orbit[-1,0])
print(golf_ts[-1])

In [ ]:
plt.axhline(0, c = "red", ls = "--", label = "flat ground")
plt.plot(golf_orbit[:,0], golf_orbit[:,1], 'k', label = "trajectory")
plt.axis("equal")
plt.title(r"Trajectory of a golf ball launched at $30^{\circ}$")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.legend()
plt.show()

In [ ]:
plt.axhline(0, c = "red", ls = "--", label = "flat ground")
for i in range(20):
    m = (i+1)*m_0 / 5
    golf_orbit, _ = leapfrog(golf, [0, 200], xs_i_golf, 0.001, golf_cond)
    plt.plot(golf_orbit[:,0], golf_orbit[:,1], 'k')
plt.plot(golf_orbit[:,0], golf_orbit[:,1], 'k', label = "trajectories")

plt.axis("equal")
plt.title(r"Trajectories of golf balls of varying masses launched at $30^{\circ}$")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.legend()
plt.show()

In [ ]:
def dist_traveled(xs_i):
    golf_orbit, _ = leapfrog(golf, [0, 200], xs_i, 0.005, golf_cond)
    return golf_orbit[-1,0]

N_mass = 400
masses = np.zeros(N_mass)
massfar = np.zeros(N_mass)
for i in range(N_mass):
    m = (i+1)*m_0 / 20
    masses[i] = m
    massfar[i] = dist_traveled(xs_i_golf)

In [ ]:
plt.plot(masses, massfar, 'k')
plt.title(r"How far does the golf ball go, as a function of mass?")
plt.xlabel("golf ball mass [kg]")
plt.ylabel("distance traveled [m]")
plt.show()

In [ ]:
def launch(theta):
    return np.array([0, 0, 171.5*np.cos(theta), 171.5*np.sin(theta)])

In [ ]:
def find_root(f, x1, x2, tol=1e-4): # this is my root-finding algorithm from pset 2!
    if f(x1):
        xtrue, xfalse = x1, x2
    else:
        xtrue, xfalse = x2, x1
    assert f(xtrue) & ~f(xfalse)
    #print('('+str(xfalse)+', '+str(xtrue)+')')
    xmid = (xtrue + xfalse)/2
    if (np.abs(xtrue - xfalse) < tol):
        return xmid
    elif f(xmid):
        return find_root(f, xmid, xfalse)
    else:
        return find_root(f, xmid, xtrue)

def angle_search(theta):
    delta=1e-3
    return (dist_traveled(launch(theta+delta)) > dist_traveled(launch(theta-delta)))

In [ ]:
m = m_0
max_angle = find_root(angle_search, np.pi / 30, np.pi / 2.5)
print(max_angle)
print(dist_traveled(launch(max_angle)))

### Problem 3.

I've plotted the required plot, and also a zoomed in version!

In [ ]:
def map(x, r):
    return r * x * (1 - x)

rs = np.arange(2.5, 4.0, 0.0004)
xs_log = 0.5 * np.ones(rs.shape[0])
for i in range(1000):
    xs_log = map(xs_log, rs)
xs_data = np.zeros(rs.shape[0]*100)
rs_data = np.zeros(rs.shape[0]*100)
for j in range(100):
    xs_log = map(xs_log, rs)
    xs_data[j*rs.shape[0]:(j+1)*rs.shape[0]] = xs_log
    rs_data[j*rs.shape[0]:(j+1)*rs.shape[0]] = rs

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rs_data, xs_data, "ok", ms=0.1, antialiased=False)
ax.set_xlabel("r")
ax.set_ylabel("x")
ax.set_title("Logistic Map!")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rs_data, xs_data, "ok", ms=0.1, antialiased=False)
ax.set_xlim(3.5, 4)
ax.set_xlabel("r")
ax.set_ylabel("x")
ax.set_title("Logistic Map, zoomed in!")
plt.show()

### Problem 4.

_(a)_ The first plot is the orbits with initial conditions and values as specified, the second plot is slices in phase space for $\Omega=0$ (they're all ellipses, as promised), and the third plot is orbits with the prettier initial conditions that Hogg used (from his github).

_(b)_ The fourth plot is the surface of section for the specified initial conditions and values, the fifth plot is the surface of section for $\Omega=0$ (they all punch through $z=0$ at about the same point, as promised), and the third plot is the surface of section for many different values of $v_{x0}$.

In [ ]:
G = 4. * np.pi**2 # a.u.^3 yr^-2 M_sun^-1 
M_sun = 1 # Solar Masses
OMEGA_SQUARED = 0.5 # units of angular frequency squared yr^-2
TIDE = np.diag([1., 1., -2.])
r0 = np.array([1., 0., 0.])
v0 = np.array([0., 2.*np.pi, 0.6])

In [ ]:
def accelerations(xs, t):
    a = 0-G * M_sun * xs[0:3] / np.linalg.norm(xs[0:3])**3 - 2. * OMEGA_SQUARED * TIDE@(xs[0:3])
    return np.concatenate((xs[3:6], a))

In [ ]:
OMEGA_SQUARED = 0.5
H_orbit, H_ts = leapfrog(accelerations, [0, 200], np.concatenate((r0, v0)), 0.0001)

In [ ]:
plt.plot(H_orbit[:,0], H_orbit[:,2], 'k-', lw=0.1, alpha=0.5)
plt.xlabel("x [a.u.]")
plt.ylabel("z [a.u.]")
plt.title(r"Orbits in $x$-$z$ with a tidal perturbation ($v_{z0}=0.6$ a.u. yr$^{-1}$)")
plt.show()

In [ ]:
OMEGA_SQUARED = 0.
H_orbit_normie, H_ts_normie = leapfrog(accelerations, [0, 200], np.concatenate((r0, v0)), 0.0001)

In [ ]:
fig, ax = plt.subplots(2, 2, sharex="col", sharey="row", layout="constrained")
ax[0,0].plot(H_orbit_normie[:,0], H_orbit_normie[:,3], 'k-', lw=0.5)
ax[0,0].set_ylabel(r"$v_x$ [a.u. yr$^{-1}$]")
ax[1,0].plot(H_orbit_normie[:,0], H_orbit_normie[:,2], 'k-', lw=0.5)
ax[1,0].set_ylabel(r"$z$ [a.u.]")
ax[1,0].set_xlabel(r"$x$ [a.u.]")
ax[1,1].set_xlabel(r"$v_z$ [a.u. yr$^{-1}$]")
ax[0,1].plot(H_orbit_normie[:,5], H_orbit_normie[:,3], 'k-', lw=0.5)
ax[1,1].plot(H_orbit_normie[:,5], H_orbit_normie[:,2], 'k-', lw=0.5)
plt.suptitle(r"Normie orbits ($\Omega = 0$ yr$^{-1}$). Everything closes!")
plt.show()

In [ ]:
OMEGA_SQUARED = 0.5
H_orbit_pretty, H_ts_pretty = leapfrog(accelerations, [0, 200], np.concatenate((r0, np.array([0., 2.*np.pi, 1.0]))), 0.0001)

In [ ]:
plt.plot(H_orbit_pretty[:,0], H_orbit_pretty[:,2], 'k-', lw=0.1, alpha=0.5)
plt.xlabel("x [a.u.]")
plt.ylabel("z [a.u.]")
plt.title(r"Orbits in $x$-$z$ with a tidal perturbation ($v_{z0}=1$ a.u. yr$^{-1}$)")
plt.show()

In [ ]:
def interpol_times(orbit, ts):
    ii = (orbit >= 0) & (np.roll(orbit, 1) < 0)
    ii[0] = False
    return np.roll(ts, 1)[ii] - np.roll(orbit, 1)[ii] * 0.0001 / (orbit[ii] - np.roll(orbit, 1)[ii])

In [ ]:
t_surf = interpol_times(H_orbit[:,2], H_ts)
t_surf_normie = interpol_times(H_orbit_normie[:,2], H_ts_normie)

In [ ]:
def get_rho_surfs(t_surf, H_ts, H_orbit):
    x_surf = np.interp(t_surf, H_ts, H_orbit[:,0])
    y_surf = np.interp(t_surf, H_ts, H_orbit[:,1])
    vx_surf = np.interp(t_surf, H_ts, H_orbit[:,3])
    vy_surf = np.interp(t_surf, H_ts, H_orbit[:,4])
    rho_surf = np.sqrt(x_surf**2 + y_surf**2)
    vrho_surf = (x_surf * vx_surf + y_surf * vy_surf) / rho_surf
    return rho_surf, vrho_surf

In [ ]:
rho_surf, vrho_surf = get_rho_surfs(t_surf, H_ts, H_orbit)
rho_surf_normie, vrho_surf_normie = get_rho_surfs(t_surf_normie, H_ts_normie, H_orbit_normie)

In [ ]:
plt.scatter(rho_surf, vrho_surf, s = 1, c = "k", marker = ".", alpha = 0.75)
plt.xlabel(r"$\rho$ [a.u.]")
plt.ylabel(r"$v_{\rho}$ [a.u. yr$^{-1}$]")
plt.title("Surface of section with perturbation ($v_{z0}=0.6$ a.u. yr$^{-1}$)")
plt.show()

In [ ]:
plt.scatter(rho_surf_normie, vrho_surf_normie, s = 1, c = "k", marker = ".", alpha = 0.75)
plt.xlabel(r"$\rho$ [a.u.]")
plt.ylabel(r"$v_{\rho}$ [a.u. yr$^{-1}$]")
plt.title("Surface of section for a normie orbit ($\Omega = 0$ yr$^{-1}$)")
plt.xlim(0.9, 1.1)
plt.ylim(-0.1, 0.1)
plt.show()

In [ ]:
def plot_one_orbit(vx0, vz0):
    H_orbit, H_ts = leapfrog(accelerations, [0, 200], np.concatenate((r0, np.array([vx0, 2.*np.pi, vz0]))), 0.0001)
    t_surf = interpol_times(H_orbit[:,2], H_ts)
    rho_surf, vrho_surf = get_rho_surfs(t_surf, H_ts, H_orbit)
    plt.scatter(rho_surf, vrho_surf, s = 1, c = "k", marker = ".", alpha = 0.65)

In [ ]:
for vx0 in np.arange(0., 1.69, 0.02):
    plot_one_orbit(vx0, 0.6)

plt.xlabel(r"$\rho$ [a.u.]")
plt.ylabel(r"$v_{\rho}$ [a.u. yr$^{-1}$]")
plt.title("Surface of section for many orbits!")
plt.savefig("surface_of_section.svg", format="svg")
plt.savefig("surface_of_section.png", format="png")
plt.show()

### Credits

Huge props to the numpy and matplotlib documentation. Also, credit to the wikipedia pages "Speed of Sound," "Golf Ball," and "Logistic Map," and to the papers "Drag Coefficients of Golf Balls" (Jenkins et al) and "A Study of Golf Ball Aerodynamic Drag" (Alam et al). Finally, credit to David Hogg's Github(tm)! All of the above code is completely my own; I did however learn that np.roll and np.interp exist, and that I should use them.